# Pipeline.ipynb — Inferensi: YOLO + PaddleOCR
Environment: `.venv312` (Python 3.12, PaddleOCR). Lihat `../INSTALL.md`.
Bagian eksperimen (OCR crop kamera jalan, dataset Denny) dipertahankan sebagai **bukti proses/kegagalan** untuk bimbingan. Implementasi final ada di Code\slpr.py. Sel di notebook ini adalah versi awal dan eksperimen, disimpan sebagai referensi dan bukti proses.

> ⚠️ Sel bertanda *hanya untuk dilihat* membaca folder yang sudah dihapus (`crops`, `dataset_denny_v2`, `crops_denny*`, `cek_label`). Outputnya adalah bukti; jangan dijalankan ulang.

## Peta isi notebook (Pipeline.ipynb, `.venv312` Python 3.12)

| Bagian | Isi | Status |
|---|---|---|
| Cek environment | 0. Cek environment | dipakai |
| Eksperimen OCR 1 (kamera jalan, gagal) | Eksperimen 1 — OCR pada crop kamera jalan | hanya untuk dilihat |
| Eksperimen OCR 2 (dataset Denny, margin, cek label, gambar penuh) | Eksperimen 2 — Dataset Denny, sampai OCR dari gambar penuh | hanya untuk dilihat |
| Uji foto asli, model lama | Uji foto asli dengan model lama | hanya untuk dilihat |
| Pipeline final (model, OCR, `read_plate`) | Pipeline final — model, OCR, `read_plate` | referensi (logika final ada di `slpr.py`) |
| Uji model gabungan + OCR pada foto asli | Uji model gabungan + OCR pada foto asli, Perbarui pola `PLAT` | referensi |

Percobaan margin crop, `--stop-votes`, dan lebar OCR **tidak** ada di notebook ini; hasilnya berasal dari run `Code\slpr.py` di folder `hasil_*`.

Output sel jangan dihapus karena merupakan bukti hasil.

## 0. Cek environment
Python 3.12 dan CUDA.

In [1]:
import sys, torch
print(sys.version)
print("cuda:", torch.cuda.is_available())

3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
cuda: True


> **Hanya untuk dilihat, jangan dijalankan ulang.** Eksperimen ini memakai `../crops` yang sudah tidak ada.

## Eksperimen 1 — OCR pada crop kamera jalan (GAGAL)
> ⚠️ **Hanya untuk dilihat (output sebagai bukti), jangan dijalankan ulang** — membaca folder yang sudah dihapus: `../crops`.

PaddleOCR dijalankan pada crop plat dari dataset kamera jalan. Hasilnya banyak kosong (`[]`) atau salah baca karena crop terlalu kecil dan buram (tinggi ±26–41 px). Preprocessing (resize, CLAHE, rec-only) di sel berikutnya tidak menyelesaikan masalah ini.

In [3]:
import cv2
from pathlib import Path
from paddleocr import PaddleOCR

ocr = PaddleOCR(
    lang="en",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
)

crops_dir = Path("../crops").resolve()
for p in sorted(crops_dir.glob("*.jpg")):
    img = cv2.imread(str(p))
    big = cv2.resize(img, None, fx=4, fy=4, interpolation=cv2.INTER_CUBIC)
    big = cv2.copyMakeBorder(big, 20, 20, 20, 20, cv2.BORDER_REPLICATE)
    result = ocr.predict(big)
    texts, scores = [], []
    for r in result:
        texts += list(r.get("rec_texts", []))
        scores += [round(float(s), 3) for s in r.get("rec_scores", [])]
    print(p.name[:12], img.shape[:2], "->", texts, scores)

c:\Users\riizk\Documents\3.Collage\SKRIPSI\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\riizk\Documents\3.Collage\SKRIPSI\.venv312\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv6_medium_det', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-OCRv6_medium_det), the model files will be automatically downloaded and saved in `C:\Users\riizk\.paddlex\official_models\PP-OCRv6_medium_det`.
Fetchin

1657251475.r (41, 101) -> ['1002'] [0.948]
1657251488.r (27, 74) -> [] []
1657251540.r (31, 82) -> ['BA1762CU'] [0.658]
1657251693.r (31, 75) -> [] []
1657251759.r (26, 67) -> [] []


In [4]:
import cv2
from paddleocr import TextRecognition

rec = TextRecognition(model_name="en_PP-OCRv5_mobile_rec")

def prep(img, scale=6):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    g = cv2.resize(g, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(4, 4))
    g = clahe.apply(g)
    g = cv2.copyMakeBorder(g, 20, 20, 20, 20, cv2.BORDER_REPLICATE)
    return cv2.cvtColor(g, cv2.COLOR_GRAY2BGR)

def run_full(img):
    texts, scores = [], []
    for r in ocr.predict(img):
        texts += list(r.get("rec_texts", []))
        scores += [round(float(s), 3) for s in r.get("rec_scores", [])]
    return texts, scores

def run_rec(img):
    out = []
    for r in rec.predict(input=img):
        out.append((r.get("rec_text"), round(float(r.get("rec_score", 0)), 3)))
    return out

for p in sorted(crops_dir.glob("*.jpg")):
    img = cv2.imread(str(p))
    pre = prep(img)
    print(p.name[:12])
    print("  full + CLAHE:", run_full(pre))
    print("  rec-only    :", run_rec(pre))


Using official model (en_PP-OCRv5_mobile_rec), the model files will be automatically downloaded and saved in `C:\Users\riizk\.paddlex\official_models\en_PP-OCRv5_mobile_rec`.
Fetching 6 files: 100%|██████████| 6/6 [00:02<00:00,  2.88it/s]


1657251475.r
  full + CLAHE: (['1 102 2J'], [0.812])
  rec-only    : [('', 0.0)]
1657251488.r
  full + CLAHE: (['1'], [0.986])
  rec-only    : [('0 1041 A3H', 0.73)]
1657251540.r
  full + CLAHE: (['BAL1762', 'C'], [0.671, 0.517])
  rec-only    : [('BA 1783 0', 0.613)]
1657251693.r
  full + CLAHE: ([], [])
  rec-only    : [('761', 0.48)]
1657251759.r
  full + CLAHE: ([], [])
  rec-only    : [('1', 0.176)]


> **Hanya untuk dilihat, jangan dijalankan ulang.** Eksperimen ini memakai `../dataset_denny_v*`, `../crops_denny` dan `../crops_denny_gt` yang sudah tidak ada.

## Eksperimen 2 — Dataset Denny (foto plat lebih besar)
> ⚠️ **Hanya untuk dilihat (output sebagai bukti), jangan dijalankan ulang** — membaca folder yang sudah dihapus: `../dataset_denny_v*`.

Model awal (`slpr_v8n_run1`) dicoba pada dataset Denny: **0 deteksi** pada banyak gambar (sel ini dan sel pengecekan conf=0.05 di bawahnya). Dilanjutkan dengan OCR pada crop dari label ground truth untuk menguji OCR terpisah dari detektor.

In [5]:
import cv2, glob
from pathlib import Path
from ultralytics import YOLO
from paddleocr import PaddleOCR

root = Path(sorted(glob.glob("../dataset_denny_v*"))[0]).resolve()
split = "test" if (root / "test" / "images").exists() else "valid"
imgs = sorted((root / split / "images").iterdir())[:12]
print(root.name, split, len(imgs), "gambar")

model = YOLO(str(Path("../runs/slpr_v8n_run1/weights/best.pt").resolve()))
ocr = PaddleOCR(lang="en", use_doc_orientation_classify=False,
                use_doc_unwarping=False, use_textline_orientation=False)

out = Path("../crops_denny")
out.mkdir(exist_ok=True)

for p in imgs:
    res = model.predict(str(p), conf=0.25, device=0, verbose=False)[0]
    for i, b in enumerate(res.boxes.xyxy.cpu().numpy().astype(int)):
        x1, y1, x2, y2 = b
        crop = res.orig_img[max(y1, 0):y2, max(x1, 0):x2]
        cv2.imwrite(str(out / f"{p.stem[:20]}_{i}.jpg"), crop)
        big = cv2.resize(crop, None, fx=4, fy=4, interpolation=cv2.INTER_CUBIC)
        big = cv2.copyMakeBorder(big, 20, 20, 20, 20, cv2.BORDER_REPLICATE)
        texts, scores = [], []
        for r in ocr.predict(big):
            texts += list(r.get("rec_texts", []))
            scores += [round(float(s), 3) for s in r.get("rec_scores", [])]
        print(p.name[:20], crop.shape[:2], "->", texts, scores)

Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\riizk\.paddlex\official_models\PP-OCRv6_medium_det`.


dataset_denny_v2 test 12 gambar


Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\riizk\.paddlex\official_models\PP-OCRv6_medium_rec`.


In [6]:
for p in imgs:
    r = model.predict(str(p), conf=0.05, device=0, verbose=False)[0]
    h, w = r.orig_shape
    lbl = root / split / "labels" / (p.stem + ".txt")
    gt = [l.split() for l in lbl.read_text().splitlines()] if lbl.exists() else []
    print(p.name[:20], f"{w}x{h}",
          "| deteksi:", len(r.boxes), [round(float(c), 2) for c in r.boxes.conf.cpu().numpy()],
          "| lebar plat asli (relatif):", [round(float(g[3]), 2) for g in gt])

plat-104-_jpg.rf.982 329x859 | deteksi: 0 [] | lebar plat asli (relatif): [0.69]
plat-105-_jpg.rf.558 3264x2448 | deteksi: 0 [] | lebar plat asli (relatif): [0.82]
plat-113-_jpg.rf.6bb 3264x2448 | deteksi: 0 [] | lebar plat asli (relatif): [0.13]
plat-12-_jpg.rf.79bd 4000x3000 | deteksi: 0 [] | lebar plat asli (relatif): [0.68]
plat-121-_jpg.rf.c95 1040x780 | deteksi: 0 [] | lebar plat asli (relatif): [0.76]
plat-16-_jpg.rf.5255 4000x3000 | deteksi: 0 [] | lebar plat asli (relatif): [0.48]
plat-171-_jpg.rf.d30 4000x3000 | deteksi: 1 [0.07] | lebar plat asli (relatif): [0.4]
plat-183-_jpg.rf.23c 3264x2448 | deteksi: 0 [] | lebar plat asli (relatif): [0.2]
plat-200-_jpg.rf.720 4000x3000 | deteksi: 0 [] | lebar plat asli (relatif): [0.54]
plat-208-_jpg.rf.0de 4000x3000 | deteksi: 0 [] | lebar plat asli (relatif): [0.54]
plat-209-_jpg.rf.70d 4000x3000 | deteksi: 0 [] | lebar plat asli (relatif): [0.47]
plat-210-_jpg.rf.0bc 3264x2448 | deteksi: 0 [] | lebar plat asli (relatif): [0.81]


### OCR pada crop ground truth + fungsi `read_plate`
> ⚠️ **Hanya untuk dilihat (output sebagai bukti), jangan dijalankan ulang** — membaca folder yang sudah dihapus: `../dataset_denny_v*`, `../crops_denny_gt`.

Menguji OCR dengan crop dari label asli, lalu menyusun `read_plate` (gabung baris atas/bawah plat) dan pola regex `PLAT` untuk validasi format.

In [7]:
import cv2
from pathlib import Path

out = Path("../crops_denny_gt")
out.mkdir(exist_ok=True)

def crop_gt(img_path, margin=0.05):
    img = cv2.imread(str(img_path))
    H, W = img.shape[:2]
    lbl = root / split / "labels" / (img_path.stem + ".txt")
    crops = []
    for line in lbl.read_text().splitlines():
        parts = line.split()
        xc, yc, bw, bh = [float(v) for v in parts[1:5]]
        x1 = int((xc - bw / 2 - margin * bw) * W)
        x2 = int((xc + bw / 2 + margin * bw) * W)
        y1 = int((yc - bh / 2 - margin * bh) * H)
        y2 = int((yc + bh / 2 + margin * bh) * H)
        crops.append(img[max(y1, 0):min(y2, H), max(x1, 0):min(x2, W)])
    return crops

for p in imgs:
    for i, crop in enumerate(crop_gt(p)):
        scale = 640 / crop.shape[1]
        interp = cv2.INTER_CUBIC if scale > 1 else cv2.INTER_AREA
        crop = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=interp)
        cv2.imwrite(str(out / f"{p.stem[:20]}_{i}.jpg"), crop)
        big = cv2.copyMakeBorder(crop, 20, 20, 20, 20, cv2.BORDER_REPLICATE)
        texts, scores = [], []
        for r in ocr.predict(big):
            texts += list(r.get("rec_texts", []))
            scores += [round(float(s), 3) for s in r.get("rec_scores", [])]
        print(p.name[:20], "->", texts, scores)

plat-104-_jpg.rf.982 -> ['2148 BL', '05·26'] [0.986, 0.997]
plat-105-_jpg.rf.558 -> ['2169 QB', '04.27'] [0.995, 0.986]
plat-113-_jpg.rf.6bb -> ['H'] [0.998]
plat-12-_jpg.rf.79bd -> ['AB 5896 WX', 'HOND', '06·24'] [0.972, 0.997, 0.992]
plat-121-_jpg.rf.c95 -> ['2441 WB', '09.26', 'D.O.R', 'PHOTOGRAP'] [0.989, 0.997, 1.0, 0.999]
plat-16-_jpg.rf.5255 -> ['HONDA', 'AD 2832 N', '01·2'] [0.999, 0.962, 0.923]
plat-171-_jpg.rf.d30 -> ['H', '3326', '02'] [1.0, 1.0, 1.0]
plat-183-_jpg.rf.23c -> ['H', '3', 'C'] [1.0, 0.578, 0.889]
plat-200-_jpg.rf.720 -> ['STANLEY V4231', '3919 JK', '06·25'] [0.992, 0.961, 0.974]
plat-208-_jpg.rf.0de -> ['H 4096 GK', '08·26'] [0.981, 0.901]
plat-209-_jpg.rf.70d -> ['4127', 'H', '09.'] [1.0, 0.996, 0.98]
plat-210-_jpg.rf.0bc -> ['4156', '10·23'] [0.967, 0.884]


In [8]:
import re
import cv2
import numpy as np
from pathlib import Path

PLAT = re.compile(r"^([A-Z]{1,2})(\d{1,4})([A-Z]{0,3})$")

def read_plate(img, ytol=0.6):
    items = []
    for r in ocr.predict(img):
        for t, s, poly in zip(r.get("rec_texts", []), r.get("rec_scores", []), r.get("rec_polys", [])):
            t = re.sub(r"[^A-Z0-9]", "", str(t).upper())
            if not t:
                continue
            p = np.array(poly)
            y1, y2 = float(p[:, 1].min()), float(p[:, 1].max())
            items.append({"t": t, "s": float(s), "x": float(p[:, 0].min()),
                          "yc": (y1 + y2) / 2, "h": y2 - y1})
    if not items:
        return None, False, []
    items.sort(key=lambda d: d["yc"])
    lines, cur = [], [items[0]]
    for it in items[1:]:
        if abs(it["yc"] - cur[-1]["yc"]) < ytol * max(it["h"], cur[-1]["h"]):
            cur.append(it)
        else:
            lines.append(cur)
            cur = [it]
    lines.append(cur)
    cands = []
    for ln in lines:
        ln.sort(key=lambda d: d["x"])
        txt = "".join(d["t"] for d in ln)
        h = sum(d["h"] for d in ln) / len(ln)
        cands.append((txt, h))
    for txt, h in cands:
        m = PLAT.match(txt)
        if m:
            return " ".join(g for g in m.groups() if g), True, cands
    best = max(cands, key=lambda c: c[1])
    return best[0], False, cands

crops_gt = Path("../crops_denny_gt").resolve()
for p in sorted(crops_gt.glob("*.jpg")):
    img = cv2.imread(str(p))
    img = cv2.copyMakeBorder(img, 20, 20, 20, 20, cv2.BORDER_REPLICATE)
    plate, ok, cands = read_plate(img)
    print(p.name[:20], "->", plate, "| valid:", ok, "| semua baris:", [c[0] for c in cands])

plat-104-_jpg.rf.982 -> 2148BL | valid: False | semua baris: ['2148BL', '0526']
plat-105-_jpg.rf.558 -> 2169QB | valid: False | semua baris: ['2169QB', '0427']
plat-113-_jpg.rf.6bb -> H | valid: False | semua baris: ['H']
plat-12-_jpg.rf.79bd -> AB 5896 WX | valid: True | semua baris: ['AB5896WX', '0624']
plat-121-_jpg.rf.c95 -> 2441WB | valid: False | semua baris: ['2441WB', '0926', 'DORPHOTOGRAP']
plat-16-_jpg.rf.5255 -> AD 2832 N | valid: True | semua baris: ['HONDA', 'AD2832N', '012']
plat-171-_jpg.rf.d30 -> H 3326 | valid: True | semua baris: ['H3326', '02']
plat-183-_jpg.rf.23c -> HC | valid: False | semua baris: ['HC']
plat-200-_jpg.rf.720 -> H 3919 JK | valid: True | semua baris: ['STANLEYV4231', 'H3919JK', '0625']
plat-208-_jpg.rf.0de -> H 4096 GK | valid: True | semua baris: ['H4096GK', '0826']
plat-209-_jpg.rf.70d -> H 4127 | valid: True | semua baris: ['H4127', '09']
plat-210-_jpg.rf.0bc -> 156 | valid: False | semua baris: ['156', '1023']


> **Hanya untuk dilihat, jangan dijalankan ulang.** Sel ini memakai `../dataset_denny_v*` yang sudah tidak ada.

### Uji margin crop pada kasus yang gagal
> ⚠️ **Hanya untuk dilihat (output sebagai bukti), jangan dijalankan ulang** — membaca folder yang sudah dihapus: `../dataset_denny_v*` (variabel `imgs`, `crop_gt`).

Mencoba margin 0.05/0.15/0.3 pada plat yang gagal terbaca.

In [9]:
fail = ["plat-104", "plat-105", "plat-113", "plat-121", "plat-183", "plat-210"]
for p in imgs:
    if not any(p.name.startswith(f) for f in fail):
        continue
    for m in (0.05, 0.15, 0.3):
        for crop in crop_gt(p, margin=m):
            scale = 640 / crop.shape[1]
            interp = cv2.INTER_CUBIC if scale > 1 else cv2.INTER_AREA
            crop = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=interp)
            crop = cv2.copyMakeBorder(crop, 20, 20, 20, 20, cv2.BORDER_REPLICATE)
            plate, ok, cands = read_plate(crop)
            print(p.name[:10], f"margin={m}", "->", plate, ok, [c[0] for c in cands])

plat-104-_ margin=0.05 -> 2148BL0526 False ['2148BL0526']
plat-104-_ margin=0.15 -> H2148BL0526 False ['H2148BL0526']
plat-104-_ margin=0.3 -> H2148BL0526 False ['H2148BL0526']
plat-105-_ margin=0.05 -> 2169QB False ['2169QB', '0427']
plat-105-_ margin=0.15 -> 2169QB False ['2169QB', '0427', 'TURN']
plat-105-_ margin=0.3 -> H 2169 QB True ['H2169QB', '0427', 'TURN', 'BVCK']
plat-113-_ margin=0.05 -> H False ['H']
plat-113-_ margin=0.15 -> H False ['H']
plat-113-_ margin=0.3 -> H False ['C', 'H']
plat-121-_ margin=0.05 -> 2441WB False ['2441WB', '0926', 'DORPHOTOGRAP']
plat-121-_ margin=0.15 -> 2441WB False ['2441WB', '0926', 'DORPHOTOGRAP']
plat-121-_ margin=0.3 -> H 2441 WB True ['H2441WB', '0926', 'DORPHOTOGRAP']
plat-183-_ margin=0.05 -> H 3 C True ['H3C']
plat-183-_ margin=0.15 -> 3 False ['3']
plat-183-_ margin=0.3 -> H 3 True ['H3']
plat-210-_ margin=0.05 -> 4156 False ['4156', '1023']
plat-210-_ margin=0.15 -> 4156K False ['4156K', '1023']
plat-210-_ margin=0.3 -> 4156K False ['

In [10]:
import re
import cv2
import numpy as np

PLAT = re.compile(r"^([A-Z]{1,2})(\d{1,4})([A-Z]{0,3})(\d{4})?$")

def read_plate(img, ytol=0.6):
    items = []
    for r in ocr.predict(img):
        for t, s, poly in zip(r.get("rec_texts", []), r.get("rec_scores", []), r.get("rec_polys", [])):
            t = re.sub(r"[^A-Z0-9]", "", str(t).upper())
            if not t:
                continue
            p = np.array(poly)
            y1, y2 = float(p[:, 1].min()), float(p[:, 1].max())
            items.append({"t": t, "s": float(s), "x": float(p[:, 0].min()),
                          "yc": (y1 + y2) / 2, "h": y2 - y1})
    if not items:
        return None, False, []
    items.sort(key=lambda d: d["yc"])
    lines, cur = [], [items[0]]
    for it in items[1:]:
        if abs(it["yc"] - cur[-1]["yc"]) < ytol * max(it["h"], cur[-1]["h"]):
            cur.append(it)
        else:
            lines.append(cur)
            cur = [it]
    lines.append(cur)
    cands = []
    for ln in lines:
        ln.sort(key=lambda d: d["x"])
        txt = "".join(d["t"] for d in ln)
        h = sum(d["h"] for d in ln) / len(ln)
        cands.append((txt, h))
    for txt, h in cands:
        m = PLAT.match(txt)
        if m:
            return " ".join(g for g in m.groups()[:3] if g), True, cands
    best = max(cands, key=lambda c: c[1])
    return best[0], False, cands

def prep_crop(crop, width=640):
    scale = width / crop.shape[1]
    interp = cv2.INTER_CUBIC if scale > 1 else cv2.INTER_AREA
    crop = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=interp)
    return cv2.copyMakeBorder(crop, 20, 20, 20, 20, cv2.BORDER_REPLICATE)

def plate_from_crop(crop):
    return read_plate(prep_crop(crop))

for p in imgs:
    for crop in crop_gt(p, margin=0.3):
        plate, ok, cands = plate_from_crop(crop)
        print(p.name[:10], "->", plate, "| valid:", ok, "|", [c[0] for c in cands])

plat-104-_ -> H 2148 BL | valid: True | ['H2148BL0526']
plat-105-_ -> H 2169 QB | valid: True | ['H2169QB', '0427', 'TURN', 'BVCK']
plat-113-_ -> H | valid: False | ['C', 'H']
plat-12-_j -> AB 5896 WX | valid: True | ['AB5896WX', '0624']
plat-121-_ -> H 2441 WB | valid: True | ['H2441WB', '0926', 'DORPHOTOGRAP']
plat-16-_j -> IN 5 ES | valid: True | ['IN5ES', 'AD2832NW', '0128']
plat-171-_ -> H 3326 QK | valid: True | ['H3326QK', '0227FAOLFE']
plat-183-_ -> H 3 | valid: True | ['H3']
plat-200-_ -> H 3919 JK | valid: True | ['H3919JK', '0625']
plat-208-_ -> H 4096 CK | valid: True | ['H4096CK', '0826', 'AHI']
plat-209-_ -> H 4127 YB | valid: True | ['H4127YB0922']
plat-210-_ -> 4156K | valid: False | ['4156K', '1023']


> **Hanya untuk dilihat, jangan dijalankan ulang.** Sel ini memakai `../cek_label` dan `../dataset_denny_v*` yang sudah tidak ada.

### Cek visual label Denny
> ⚠️ **Hanya untuk dilihat (output sebagai bukti), jangan dijalankan ulang** — membaca folder yang sudah dihapus: `../dataset_denny_v*`; gambar `../cek_label` sudah dihapus.

Kotak label pada dataset Denny **banyak yang meleset** dari plat sebenarnya, sehingga dataset ini tidak dipakai untuk training.

In [11]:
cek = Path("../cek_label")
cek.mkdir(exist_ok=True)
for name in ["plat-105", "plat-121", "plat-16", "plat-171"]:
    p = next(q for q in imgs if q.name.startswith(name + "-"))
    img = cv2.imread(str(p))
    H, W = img.shape[:2]
    for line in (root / split / "labels" / (p.stem + ".txt")).read_text().splitlines():
        xc, yc, bw, bh = [float(v) for v in line.split()[1:5]]
        x1, y1 = int((xc - bw / 2) * W), int((yc - bh / 2) * H)
        x2, y2 = int((xc + bw / 2) * W), int((yc + bh / 2) * H)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), max(3, W // 300))
    s = 1000 / max(H, W)
    cv2.imwrite(str(cek / (name + ".jpg")), cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA))
print("selesai")


selesai


> **Hanya untuk dilihat, jangan dijalankan ulang.** Sel ini memakai `../dataset_denny_v*` yang sudah tidak ada.

### OCR dari gambar penuh
> ⚠️ **Hanya untuk dilihat (output sebagai bukti), jangan dijalankan ulang** — membaca folder yang sudah dihapus: `../dataset_denny_v*` (variabel `imgs`).

In [12]:
def plate_from_image(path, long_side=1600):
    img = cv2.imread(str(path))
    s = long_side / max(img.shape[:2])
    img = cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    return read_plate(img)

for p in imgs:
    plate, ok, cands = plate_from_image(p)
    print(p.name[:10], "->", plate, "| valid:", ok, "|", [c[0] for c in cands][:6])

plat-104-_ -> H 2148 BL | valid: True | ['H2148BL', '0526']
plat-105-_ -> H 2169 QB | valid: True | ['H2169QB', '0427', 'BVCKTURN']
plat-113-_ -> 2296SB31 | valid: False | ['HONDA', '2296SB31', '1123']
plat-12-_j -> AB5896WXSPOLRI | valid: False | ['AB5896WXSPOLRI', 'HONDA0624']
plat-121-_ -> H 2441 WB | valid: True | ['SUPRO', 'JIP', 'H2441WB', '0926', 'IAMSORRYDOR']
plat-16-_j -> IN 5025 | valid: True | ['IN5025', 'HONDA', 'AD2832NW', '0128']
plat-171-_ -> H 3326 QK | valid: True | ['H3326QK', '0227POLPI']
plat-183-_ -> H 3520 BLG | valid: True | ['H3520BLG', '0328']
plat-200-_ -> H 3919 JK | valid: True | ['STANLEYW4231', 'H3919JK', '40625']
plat-208-_ -> H 4096 GK | valid: True | ['H4096GK', '0826']
plat-209-_ -> OOLEH41270922YBPOLRI | valid: False | ['OOLEH41270922YBPOLRI', 'IN']
plat-210-_ -> H 4156 K | valid: True | ['H4156K', '1023', 'W']


> **Hanya untuk dilihat, jangan dijalankan ulang.** Sel ini memakai `../hasil_foto.csv` (sudah tidak ada) dan model lama `slpr_v8n_run1`; `../hasil_foto` sudah dipindah ke `_sampah/`.

## Uji foto asli dengan model lama
Model lama (`slpr_v8n_run1`) pada `../foto_uji`: **0 deteksi** pada semua foto (`TIDAK TERDETEKSI`), jadi tidak ada crop untuk dibaca OCR; kolom OCR di CSV kosong karena itu, bukan karena OCR gagal. Sel ini menulis `../hasil_foto.csv` (file lama sudah dihapus karena usang). Butuh variabel `model` dari sel berikutnya (Pipeline final) — jalankan sel itu dulu jika mengulang.

In [ ]:
import csv, time
from pathlib import Path
import cv2

foto_dir = Path("../foto_uji").resolve()
files = sorted(p for p in foto_dir.iterdir()
               if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
print(len(files), "foto")

out_dir = Path("../hasil_foto")
out_dir.mkdir(exist_ok=True)

rows = []
for p in files:
    img = cv2.imread(str(p))
    if img is None:
        print("gagal baca:", p.name)
        continue
    H, W = img.shape[:2]
    t0 = time.time()
    res = model.predict(img, conf=0.25, device=0, verbose=False)[0]
    boxes = res.boxes.xyxy.cpu().numpy().astype(int)
    confs = res.boxes.conf.cpu().numpy()
    if len(boxes) == 0:
        rows.append([p.name, "", "", False, "", ""])
        print(p.name, "-> TIDAK TERDETEKSI")
        continue
    k = int(confs.argmax())
    x1, y1, x2, y2 = boxes[k]
    mx, my = int(0.3 * (x2 - x1)), int(0.3 * (y2 - y1))
    crop = img[max(y1 - my, 0):min(y2 + my, H), max(x1 - mx, 0):min(x2 + mx, W)]
    cv2.imwrite(str(out_dir / p.name), crop)
    plate, valid, cands = plate_from_crop(crop)
    lebar = round((x2 - x1) / W, 3)
    rows.append([p.name, "", plate, valid, round(float(confs[k]), 2), lebar])
    print(p.name[:14], "->", plate, "| valid:", valid, "| conf:", round(float(confs[k]), 2),
          "| lebar plat:", lebar, "| waktu:", round(time.time() - t0, 2), "s")

with open("../hasil_foto.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["file", "plat_benar", "plat_ocr", "lolos_format", "conf_yolo", "lebar_plat_rel"])
    w.writerows(rows)
print("CSV ditulis:", len(rows), "baris")

4 foto
WhatsApp Image 2026-10-05 at 17.52.20 (1).jpeg -> TIDAK TERDETEKSI
WhatsApp Image 2026-10-05 at 17.52.20 (2).jpeg -> TIDAK TERDETEKSI
WhatsApp Image 2026-10-05 at 17.52.20.jpeg -> TIDAK TERDETEKSI
WhatsApp Image 2026-10-05 at 17.52.22.jpeg -> TIDAK TERDETEKSI
CSV ditulis: 4 baris


: 

## Pipeline final — model, OCR, `read_plate`
Memuat YOLO baseline dan PaddleOCR, mendefinisikan `PLAT` dan `read_plate` yang dipakai sel selanjutnya.

In [1]:
import re, cv2, numpy as np
from pathlib import Path
from ultralytics import YOLO
from paddleocr import PaddleOCR

model = YOLO(str(Path("../runs/slpr_v8n_run1/weights/best.pt").resolve()))
ocr = PaddleOCR(lang="en", use_doc_orientation_classify=False,
                use_doc_unwarping=False, use_textline_orientation=False)

PLAT = re.compile(r"^([A-Z]{1,2})(\d{2,4})([A-Z]{0,3})(\d{4})?$")

def read_plate(img, ytol=0.6):
    items = []
    for r in ocr.predict(img):
        for t, s, poly in zip(r.get("rec_texts", []), r.get("rec_scores", []), r.get("rec_polys", [])):
            t = re.sub(r"[^A-Z0-9]", "", str(t).upper())
            if not t:
                continue
            p = np.array(poly)
            y1, y2 = float(p[:, 1].min()), float(p[:, 1].max())
            items.append({"t": t, "s": float(s), "x": float(p[:, 0].min()),
                          "yc": (y1 + y2) / 2, "h": y2 - y1})
    if not items:
        return None, False, []
    items.sort(key=lambda d: d["yc"])
    lines, cur = [], [items[0]]
    for it in items[1:]:
        if abs(it["yc"] - cur[-1]["yc"]) < ytol * max(it["h"], cur[-1]["h"]):
            cur.append(it)
        else:
            lines.append(cur)
            cur = [it]
    lines.append(cur)
    cands = []
    for ln in lines:
        ln.sort(key=lambda d: d["x"])
        cands.append(("".join(d["t"] for d in ln), sum(d["h"] for d in ln) / len(ln)))
    matches = [(h, txt, PLAT.match(txt)) for txt, h in cands if PLAT.match(txt)]
    if matches:
        h, txt, m = max(matches, key=lambda x: x[0])
        return " ".join(g for g in m.groups()[:3] if g), True, cands
    return max(cands, key=lambda c: c[1])[0], False, cands

def prep_crop(crop, width=640):
    scale = width / crop.shape[1]
    interp = cv2.INTER_CUBIC if scale > 1 else cv2.INTER_AREA
    crop = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=interp)
    return cv2.copyMakeBorder(crop, 20, 20, 20, 20, cv2.BORDER_REPLICATE)

def plate_from_crop(crop):
    return read_plate(prep_crop(crop))

print("siap")

c:\Users\riizk\Documents\3.Collage\SKRIPSI\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\riizk\Documents\3.Collage\SKRIPSI\.venv312\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\riizk\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\U

siap


> **Hanya untuk dilihat, jangan dijalankan ulang.** Sel ini menulis ke `../hasil_foto` (sudah dipindah ke `_sampah/`); menjalankannya akan membuat folder baru.

## Uji model gabungan + OCR pada foto asli
Model `slpr_v8n_gabungan_run1` mendeteksi plat pada foto uji (conf 0.91–0.93), crop dengan margin 30%, lalu OCR → gambar hasil ke `../hasil_foto`.

In [3]:
best_new = Path("../runs/slpr_v8n_gabungan_run1/weights/best.pt").resolve()
model2 = YOLO(str(best_new))

foto_dir = Path("../foto_uji").resolve()
out_dir = Path("../hasil_foto")
out_dir.mkdir(exist_ok=True)

for p in sorted(foto_dir.iterdir()):
    if p.suffix.lower() not in (".jpg", ".jpeg", ".png"):
        continue
    img = cv2.imread(str(p))
    H, W = img.shape[:2]
    res = model2.predict(img, conf=0.25, device=0, verbose=False)[0]
    vis = img.copy()
    for (x1, y1, x2, y2), c in zip(res.boxes.xyxy.cpu().numpy().astype(int),
                                   res.boxes.conf.cpu().numpy()):
        mx, my = int(0.3 * (x2 - x1)), int(0.3 * (y2 - y1))
        crop = img[max(y1 - my, 0):min(y2 + my, H), max(x1 - mx, 0):min(x2 + mx, W)]
        plate, valid, cands = plate_from_crop(crop)
        print(p.name[-18:], "| conf", round(float(c), 2), "| lebar", round((x2 - x1) / W, 3),
              "->", plate, "| valid:", valid, "|", [t for t, _ in cands])
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), max(2, W // 400))
    cv2.imwrite(str(out_dir / ("det_" + p.stem[-12:] + ".jpg")), vis)
print("selesai ->", out_dir.resolve())

 17.52.20 (1).jpeg | conf 0.93 | lebar 0.121 -> B 6229 VXD | valid: True | ['B6229VXD0527']
 17.52.20 (1).jpeg | conf 0.64 | lebar 0.074 -> 2068ACE | valid: False | ['2068ACE', '0227']
 17.52.20 (2).jpeg | conf 0.93 | lebar 0.136 -> B6229VXD05270 | valid: False | ['B6229VXD05270']
5 at 17.52.20.jpeg | conf 0.91 | lebar 0.109 -> AE 6923 DO | valid: True | ['AE6923DO', '0227']
5 at 17.52.22.jpeg | conf 0.93 | lebar 0.117 -> N 4039 ABQ | valid: True | ['N4039ABQ', '1228']
selesai -> C:\Users\riizk\Documents\3.Collage\SKRIPSI\hasil_foto


### Perbarui pola `PLAT`
Regex diperlonggar (`(\d*)` untuk suffix masa berlaku).

Pola ini versi awal (huruf seri boleh kosong). Versi final di slpr.py mewajibkan huruf seri dan memecah bacaan jadi inti dan awalan.

In [4]:
PLAT = re.compile(r"^([A-Z]{1,2})(\d{2,4})([A-Z]{0,3})(\d*)$")
print("pola diperbarui")

pola diperbarui
